### Imports

In [2]:
import os
import numpy as np
import pandas as pd

In [8]:
base_path = "/mnt/raid/emotional_data_raquel/fulldata_mine"

___________

## Check cum_dist quality

In [4]:
results = []

for root, dirs, files in os.walk(base_path):
    if "alldata.csv" in files:
        file_path = os.path.join(root, "alldata.csv")
        
        # 🔥 Extract participant + session from path
        parts = root.split(os.sep)
        
        try:
            sub = [p for p in parts if p.startswith("sub-")][0]
        except:
            sub = "unknown_sub"
            
        try:
            ses = [p for p in parts if p.startswith("ses-")][0]
        except:
            ses = "unknown_ses"
        
        label = f"{sub} | {ses}"
        
        try:
            df = pd.read_csv(file_path)
            
            has_column = "cum_dist" in df.columns
            
            if has_column:
                non_null = df["cum_dist"].notna().sum()
                has_values = non_null > 0
                percent_valid = (non_null / len(df)) * 100
            else:
                has_values = False
                percent_valid = 0
            
            results.append({
                "participant_session": label,
                "has_cum_dist_column": has_column,
                "has_values": has_values,
                "valid_%": round(percent_valid, 2)
            })
        
        except Exception as e:
            results.append({
                "participant_session": label,
                "has_cum_dist_column": False,
                "has_values": False,
                "valid_%": 0,
                "error": str(e)
            })

results_df = pd.DataFrame(results)

print(results_df)

          participant_session  has_cum_dist_column  has_values  valid_%
0    sub-OE020 | ses-Norrebro                False       False      0.0
1    sub-OE020 | ses-Nordhavn                 True        True    100.0
2    sub-OE005 | ses-Nordhavn                False       False      0.0
3    sub-OE005 | ses-Hellerup                 True        True    100.0
4    sub-OE018 | ses-Hellerup                 True        True    100.0
5   sub-OE012 | ses-Norreport                False       False      0.0
6    sub-OE021 | ses-Norrebro                False       False      0.0
7    sub-OE021 | ses-Hellerup                False       False      0.0
8   sub-OE015 | ses-Norreport                 True        True    100.0
9    sub-OE002 | ses-Hellerup                 True        True    100.0
10   sub-OE022 | ses-Norrebro                False       False      0.0
11  sub-OE022 | ses-Norreport                 True        True    100.0
12   sub-OE022 | ses-Nordhavn                 True        True  

# Define segments

In [9]:
segments_dict = {
    "Norreport": ("OE004", [
        (0, 315, "U+M"),
        (360, 410, "U+M"),
        (411, 548, "U+N+M"),
        (549, 767, "N"),
        (768, 884, "U+M"),
        (885, 990, "U+N"),
        (991, 1111, "U"),
        (1150, 1214, "U+N+M"),
        (1215, 1400, "N")
    ]),
    
    "Norrebro": ("OE022", [
        (0, 400, "U+M"),
        (401, 866, "U+M"),
        (901, 1086, "U+N+M"),
        (1087, 1323, "N"),
        (1351, 1500, "U+N")
    ]),
    
    "Nordhavn": ("OE010", [
        (0, 175, "U+M"),
        (176, 488, "U+N+M"),
        (489, 658, "U+M"),
        (659, 895, "U"),
        (896, 1045, "N"),
        (1072, 1227, "U+N"),
        (1228, 1400, "U+M")
    ]),
    
    "Hellerup": ("OE002", [
        (0, 198, "U+M"),
        (199, 355, "U+N"),
        (396, 660, "U+N+M"),
        (684, 1161, "U+M"),
        (1162, 1260, "U+N"),
        (1261, 1500, "N")
    ])
}

## Helper Functions

In [10]:
def video_to_real(video_seconds, video_start_time):
    return video_start_time + pd.Timedelta(seconds=video_seconds)

def time_to_distance(df, timestamp):
    row = df.iloc[(df['time'] - timestamp).abs().argsort()[:1]]
    return row['cum_dist'].values[0]

def is_valid_session(df):
    return ("cum_dist" in df.columns) and (df["cum_dist"].notna().any())

## Valid sessions

In [14]:
valid_sessions = []

for participant in os.listdir(base_path):

    participant_path = os.path.join(base_path, participant)

    if not os.path.isdir(participant_path):
        continue

    for session in os.listdir(participant_path):

        session_path = os.path.join(participant_path, session)

        if not os.path.isdir(session_path):
            continue

        file_path = os.path.join(session_path, "alldata.csv")

        if not os.path.exists(file_path):
            continue

        df = pd.read_csv(file_path)

        if is_valid_session(df):
            session_name = session.split("-")[1]
            participant_id = participant.replace("sub-", "")

            valid_sessions.append({
                "participant": participant_id,
                "session": session_name,
                "path": file_path
            })

valid_sessions_df = pd.DataFrame(valid_sessions)

print("\nVALID SESSIONS:")
print(valid_sessions_df)


VALID SESSIONS:
   participant    session                                               path
0        OE020   Nordhavn  /mnt/raid/emotional_data_raquel/fulldata_mine/...
1        OE005   Hellerup  /mnt/raid/emotional_data_raquel/fulldata_mine/...
2        OE018   Hellerup  /mnt/raid/emotional_data_raquel/fulldata_mine/...
3        OE015  Norreport  /mnt/raid/emotional_data_raquel/fulldata_mine/...
4        OE002   Hellerup  /mnt/raid/emotional_data_raquel/fulldata_mine/...
5        OE022  Norreport  /mnt/raid/emotional_data_raquel/fulldata_mine/...
6        OE022   Nordhavn  /mnt/raid/emotional_data_raquel/fulldata_mine/...
7        OE009   Nordhavn  /mnt/raid/emotional_data_raquel/fulldata_mine/...
8        OE019   Hellerup  /mnt/raid/emotional_data_raquel/fulldata_mine/...
9        OE023   Norrebro  /mnt/raid/emotional_data_raquel/fulldata_mine/...
10       OE023  Norreport  /mnt/raid/emotional_data_raquel/fulldata_mine/...
11       OE023   Nordhavn  /mnt/raid/emotional_data_raquel/

# Building Typology Map (per path)

In [15]:
typology_map = []

for path_name, (ref_participant, segments) in segments_dict.items():

    print(f"\nProcessing reference for {path_name}...")

    # find reference session
    ref_row = valid_sessions_df[
        (valid_sessions_df["participant"] == ref_participant) &
        (valid_sessions_df["session"] == path_name)
    ]

    if ref_row.empty:
        print(f"⚠️ Missing reference for {path_name}")
        continue

    ref_path = ref_row.iloc[0]["path"]
    df_ref = pd.read_csv(ref_path, parse_dates=["time"]).sort_values("time")

    video_start_time = df_ref["time"].iloc[0]

    for start_sec, end_sec, typology in segments:

        t_start = video_to_real(start_sec, video_start_time)
        t_end   = video_to_real(end_sec, video_start_time)

        d_start = time_to_distance(df_ref, t_start)
        d_end   = time_to_distance(df_ref, t_end)

        typology_map.append({
            "path": path_name,
            "lowerbound": d_start,
            "higherbound": d_end,
            "typology": typology
        })

typology_df = pd.DataFrame(typology_map)
typology_df = typology_df.sort_values(["path", "lowerbound"]).reset_index(drop=True)

print("\nTYPOLOGY MAP:")
print(typology_df)


Processing reference for Norreport...

Processing reference for Norrebro...
⚠️ Missing reference for Norrebro

Processing reference for Nordhavn...

Processing reference for Hellerup...

TYPOLOGY MAP:
         path  lowerbound  higherbound typology
0    Hellerup        19.0         52.4      U+M
1    Hellerup        53.6        132.5      U+N
2    Hellerup       136.3        438.8    U+N+M
3    Hellerup       439.3       1096.7      U+M
4    Hellerup      1097.9       1249.3      U+N
5    Hellerup      1250.9       1414.9        N
6    Nordhavn         0.0          0.0      U+M
7    Nordhavn         0.0        218.3    U+N+M
8    Nordhavn       219.5        415.9      U+M
9    Nordhavn       425.2        730.5        U
10   Nordhavn       732.0        825.2        N
11   Nordhavn       827.0       1023.0      U+N
12   Nordhavn      1024.2       1200.8      U+M
13  Norreport         1.6         78.4      U+M
14  Norreport        79.2        128.0      U+M
15  Norreport       129.5     

## Apply to all valid sessions

In [16]:
def assign_typology(df, path_name, typology_df):

    path_rows = typology_df[typology_df["path"] == path_name]

    def find_typology(dist):
        match = path_rows[
            (path_rows["lowerbound"] <= dist) &
            (dist < path_rows["higherbound"])
        ]
        if match.empty:
            return np.nan
        return match.iloc[0]["typology"]

    df["typology"] = df["cum_dist"].apply(find_typology)

    return df

In [17]:
# save updated files
for _, row in valid_sessions_df.iterrows():

    participant = row["participant"]
    session = row["session"]
    file_path = row["path"]

    print(f"\nApplying typology → {participant} | {session}")

    df = pd.read_csv(file_path, parse_dates=["time"])

    df = assign_typology(df, session, typology_df)

    # 🔥 Get folder path
    folder = os.path.dirname(file_path)

    # 🔥 Define NEW filename (no replacement)
    output_path = os.path.join(folder, "alldata_with_typology.csv")

    df.to_csv(output_path, index=False)

print("\nDONE 🚀")


Applying typology → OE020 | Nordhavn

Applying typology → OE005 | Hellerup

Applying typology → OE018 | Hellerup

Applying typology → OE015 | Norreport

Applying typology → OE002 | Hellerup

Applying typology → OE022 | Norreport

Applying typology → OE022 | Nordhavn

Applying typology → OE009 | Nordhavn

Applying typology → OE019 | Hellerup

Applying typology → OE023 | Norrebro

Applying typology → OE023 | Norreport

Applying typology → OE023 | Nordhavn

Applying typology → OE004 | Norreport

Applying typology → OE010 | Nordhavn

DONE 🚀
